# Curio Seeker - SC14

## Import packages

In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY
from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

In [32]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.plotting import rel_fontsize
from utils.qc import qc_metrics
from _curio import curio

## Create sdata

In [33]:
dataset_id = 'SC14'

input_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/{dataset_id}/curioseeker/results/OUTPUT/{dataset_id}'
output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/{dataset_id}/harpy/sdata.zarr'

pipeline_name = 'CurioSeeker'
species = 'human'

In [34]:
sdata = curio(
    input_path, 
    dataset_id,
    output_path
)
sdata

c:\Users\julien.mortier\OneDrive - VIB\Documents\code\2024-07-UTBenchmark-SpC\Curio_seeker\harpy\spatialdata_curio.py:63: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  metrics = pd.read_csv(path / file_names[CurioKeys.METRICS_FILE], sep=r"\,", header=0)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\_elements.py:105: UserWarning: Key `SC14` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
2025-08-22 15:21:06,297 - harpy.image._manager - INFO - Writing results to layer 'SC14_labels'


INFO     The Zarr backing store has been changed from None the new file path:                                      
         E:\2024-07-UTBenchmark-SpC\data\processed\Curio_seeker\SC14\harpy\sdata.zarr                              


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: E:\2024-07-UTBenchmark-SpC\data\processed\Curio_seeker\SC14\harpy\sdata.zarr
├── Labels
│     └── 'SC14_labels': DataArray[yx] (4757, 4710)
├── Shapes
│     └── 'SC14': GeoDataFrame shape: (26901, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (26901, 23297)
with coordinate systems:
    ▸ 'SC14', with elements:
        SC14_labels (Labels), SC14 (Shapes)

## Specify plotting parameters

In [35]:
DPI = 100
MAX_DIM_FOR_PLOTS = 5000

figsize_factor = DPI
figsize = (sdata.labels[f'{dataset_id}_labels'].shape[1]/figsize_factor, sdata.labels[f'{dataset_id}_labels'].shape[0]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

figsize: (47.1, 47.57)
plots_path: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots


In [36]:
# Expand labels layer masks for plotting (otherwise, the plots are not very clear)
sdata = hp.im.expand_labels_layer(
    sdata,
    labels_layer=f"{dataset_id}_labels",
    distance=3,
    output_labels_layer=f"{dataset_id}_expanded_labels",
    output_shapes_layer=f"{dataset_id}_expanded",
    overwrite=True,
)

# Add table layer annotated by expanded labels layer
adata = sdata.tables['table'].copy()
adata.obs[_REGION_KEY] = f"{dataset_id}_expanded_labels"

sdata = hp.tb.add_table_layer(
    sdata,
    adata=adata,
    output_layer='table_plots',
    region=f"{dataset_id}_expanded_labels",
    overwrite=True,
)


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-08-22 15:21:51,769 - harpy.image.segmentation._map - INFO - Linking labels across chunks.
2025-08-22 15:21:51,795 - harpy.image._manager - INFO - Writing results to layer 'SC14_expanded_labels'
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-08-22 15:21:55,259 - harpy.shape._manager - INFO - Finished vectorizing. Dissolving shapes at the border of the chunks. This can take a couple minutes if input mask contains a lot of chunks.
2025-08-22 15:21:56,224 - harpy.shape._manager - INFO - Dissolve is done.
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\models\models.py:1144: UserWarning: Converting `

## Plot Curio Seeker columns

In [37]:
# Plot Curio Seeker numerical columns
for column in ['nCount_RNA', 'nFeature_RNA', 'numReads', 'log10_numReads', 'log10_nCount_RNA', 'log10_nFeature_RNA', 'percent.rp', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'log_umi']:
    
    values = sdata.tables["table_plots"].obs[column].to_numpy()
    vmin, vmax = np.percentile(values, [1, 99])
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)    
    
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - {column}"

    sdata.pl.render_labels(
        element=f"{dataset_id}_expanded_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=column, table_name='table_plots', cmap="viridis", norm=norm
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=True,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
    
    cbar = fig.axes[-1]
    cbar.set_ylabel(column, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
    cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'{pipeline_name}_{column}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_nCount_RNA.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_nFeature_RNA.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_numReads.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_log10_numReads.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_log10_nCount_RNA.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_log10_nFeature_RNA.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_percent.rp.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_percent.mt.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/

In [38]:
# Plot Curio Seeker categorical columns
for column in ['SCT_snn_res.0.2', 'seurat_clusters', 'cluster']:
    with matplotlib.rc_context({'legend.markerscale': 10}):
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} - {column}"

        sdata.pl.render_labels(
            element=f"{dataset_id}_expanded_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=column, table_name='table_plots', cmap="rainbow"
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            legend_fontsize=rel_fontsize(fig, 0.015), 
            legend_fontweight='bold', 
            legend_loc='right margin',
            na_in_legend=False,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'{pipeline_name}_{column}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)


Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_SCT_snn_res.0.2.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_seurat_clusters.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\CurioSeeker_cluster.png


## Calculate pixel size

In [39]:
# Determine pixel size in microns
from math import dist

union_geom = sdata.shapes[dataset_id].union_all()
bbox = union_geom.minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area_bbox',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1])
side2 = dist(coords[1], coords[2])

microns_per_pixel_x = 3000 / side1
microns_per_pixel_y = 3000 / side2

print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to summise what the 'pixel size' is, but it should be good enough. If the sides of the capture area are not exactly 3 mm.

Pixel size: 0.636 µm (X), 0.633 µm (Y)
Pixel size: 0.635 µm


In [40]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bbox"

sdata.pl.render_shapes(
    element=dataset_id, color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area_bbox', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' do disable this    
         behaviour.                                                                                                
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\capture_area_bbox.png


## Create bounding box for total capture area

In [41]:
# Add bounding box for total capture area
union_geom = sdata.shapes[dataset_id].union_all()
hull = union_geom.convex_hull

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[hull]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

In [42]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area convex hull"

sdata.pl.render_shapes(
    element=dataset_id, color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_hull.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' do disable this    
         behaviour.                                                                                                
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\capture_area_hull.png


In [43]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

# NOTE: This is somewhat biased because we are using the size of the capture area to determine the pixel size. However, it should be close.

Area of capture area: 8.53 mm²


In [44]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size

Dimensions of capture area bbox: 0.30 cm x 0.30 cm


In [45]:
# Total number of spots in table
n_spots = len(sdata.tables['table'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 26901


In [46]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables['table'].obs[sdata.tables['table'].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

Number of spots in table with at least one UMI: 26901


In [47]:
metrics = pd.read_csv(os.path.join(input_path, f'{dataset_id}_Metrics.csv'))
metrics

,Unnamed: 0,Values,Category
0,Total_raw_fastq_readpairs,1.609948e+08,read_quality
1,Readpairs_proper_structure,1.140148e+08,read_quality
2,Readpairs_with_adapter,2.561578e+06,read_quality
3,Percent_readpairs_proper_structure,7.082000e+01,read_quality
4,Percent_readpairs_with_adapter,1.590000e+00,read_quality
5,Percent_adapter_free_readpairs_proper_structure,7.196000e+01,read_quality
6,Total_reads_in_genes,1.859422e+07,read_quality
7,Percent_reads_in_genes,1.155000e+01,read_quality
8,Total_barcodes_on_substrate,7.122300e+04,bead_matching_quality
9,Total_barcodes_recovered,2.690100e+04,bead_matching_quality


In [48]:
# Total number of spots in capture area
n_spots_capture_area = metrics.loc[metrics["Unnamed: 0"] == "Total_barcodes_on_substrate", "Values"].iloc[0]
print(f"Total number of spots in capture area: {n_spots_capture_area:.0f}")

Total number of spots in capture area: 71223


## Capture effeciency metrics

In [49]:
# Total number of genes
print(f"Total number of genes: {(sdata.tables['table'].X.sum(axis=0) > 0).sum()}")

Total number of genes: 22303


In [50]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['table'].X.sum()}")

Total number of unique UMIs: 579472.0


In [51]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['table'].obs['nCount_RNA'].sum()}")

Total number of unique UMIs: 579472.0


In [52]:
# Tot number of UMIs
print(f"Total number of UMIs: {sdata.tables['table'].obs['numReads'].sum()}")

Total number of UMIs: 12360822


## QC

In [53]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
sdata = qc_metrics(
    sdata,
    'table',
    species
)

# Update table for plotting
sdata = qc_metrics(
    sdata,
    'table_plots',
    species
)

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SC14_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SC14_expanded_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
2025-08-22 15:25:17,991 - harpy.utils._io - WARNING - layer with name 'table' already exists. Overwriting...
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SC14_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SC14_expanded_labels', which is not 

In [54]:
# Plot QC metrics
for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:

    values = sdata.tables["table_plots"].obs[qc_metric].to_numpy()
    vmin, vmax = np.percentile(values, [1, 99])
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)    
    
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"scanpy - {qc_metric}"

    sdata.pl.render_labels(
        element=f"{dataset_id}_expanded_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, table_name='table_plots', cmap="viridis", norm=norm
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=True,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
    
    cbar = fig.axes[-1]
    cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
    cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'scanpy_{qc_metric}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\scanpy_n_genes_by_counts.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\scanpy_total_counts.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\scanpy_pct_counts_in_top_50_genes.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\scanpy_total_counts_mt.png
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy\plots\scanpy_pct_counts_mt.png


## Export unique UMI counts per gene

In [55]:
gene_counts = np.array(sdata.tables['table'].X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": sdata.tables['table'].var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / capture_area
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)